**1.Cài đặt thư viện**

In [ ]:
%pip install pymysql pandas numpy faker

  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached faker-40.36.0-py3-none-any.whl.metadata (16 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.0 MB)
Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached faker-40.36.0-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.


**2.Import thư viện và kết nối database**

In [ ]:
import pymysql
import pandas as pd
import numpy as np
import time
import random
from faker import Faker


try:
    conn = pymysql.connect(
        host='localhost',
        user='root',
        password='123456',
        database='sql_predictor_test',
        autocommit=True
    )
    cursor = conn.cursor()
    print("Kết nối MariaDB thành công!")
except Exception as e:
    print(f"Lỗi kết nối: {e}")

Kết nối MariaDB thành công!


**3.Khởi tạo dữ liệu và bảng**

In [3]:
cursor.execute("DROP TABLE IF EXISTS customer_transactions;")
cursor.execute("""
CREATE TABLE customer_transactions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    customer_name VARCHAR(100),
    age INT,
    transaction_amount DECIMAL(10, 2),
    transaction_date DATE,
    status VARCHAR(20)
);
""")
print("Đã tạo bảng customer_transactions!")

Đã tạo bảng customer_transactions!


**4.Gen dữ liệu giả**

In [ ]:
fake = Faker()

num_records = 100000
data_to_insert = []

for _ in range(num_records):
    name = fake.name()
    age = random.randint(18, 80)
    amount = round(random.uniform(10.0, 5000.0), 2)
    date = fake.date_between(start_date='-5y', end_date='today')
    status = random.choice(['pending', 'completed', 'failed', 'refunded'])
    
    data_to_insert.append((name, age, amount, date, status))


insert_sql = """
INSERT INTO customer_transactions (customer_name, age, transaction_amount, transaction_date, status)
VALUES (%s, %s, %s, %s, %s)
"""
cursor.executemany(insert_sql, data_to_insert)
print(f"Đã chèn thành công {num_records} dòng dữ liệu mẫu!")

Đã chèn thành công 1000000 dòng dữ liệu mẫu!


**5.Truy vấn và thu thập log**

In [6]:
import re

log_data = []

queries_to_test = [
    "SELECT * FROM customer_transactions WHERE age > 30;",
    "SELECT status, COUNT(*) FROM customer_transactions GROUP BY status;",
    "SELECT * FROM customer_transactions ORDER BY transaction_amount DESC LIMIT 100;",
    "SELECT AVG(transaction_amount) FROM customer_transactions WHERE status = 'completed';",
    "SELECT * FROM customer_transactions WHERE transaction_amount BETWEEN 100 AND 500;",
    "SELECT customer_name FROM customer_transactions WHERE customer_name LIKE '%Smith%';",
    "SELECT * FROM customer_transactions WHERE age > 20 AND status = 'pending' ORDER BY age ASC;"
]


for _ in range(300):
    query = random.choice(queries_to_test)
    query_upper = query.upper()
    query_length = len(query)
    has_where = 1 if 'WHERE' in query_upper else 0
    has_group_by = 1 if 'GROUP BY' in query_upper else 0
    has_order_by = 1 if 'ORDER BY' in query_upper else 0
    has_like = 1 if 'LIKE' in query_upper else 0

    start_time = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    end_time = time.time()
    
    execution_time_ms = (end_time - start_time) * 1000
    row_count = len(results)

    is_slow = 1 if execution_time_ms > 15 else 0
    
    log_data.append({
        'query_string': query,
        'query_length': query_length,
        'has_where': has_where,
        'has_group_by': has_group_by,
        'has_order_by': has_order_by,
        'has_like': has_like,
        'row_count': row_count,
        'execution_time_ms': execution_time_ms,
        'is_slow_query': is_slow
    })

df_logs = pd.DataFrame(log_data)
df_logs.to_csv('dataset_raw.csv', index=False)
print("Đã cập nhật dữ liệu chuẩn vào dataset_raw.csv!")
df_logs.head()

Đã cập nhật dữ liệu chuẩn vào dataset_raw.csv!


,query_string,query_length,has_where,has_group_by,has_order_by,has_like,row_count,execution_time_ms,is_slow_query
0,SELECT * FROM customer_transactions WHERE age ...,91,1,0,1,0,237836,1666.270733,1
1,SELECT * FROM customer_transactions WHERE age ...,51,1,0,0,0,793258,5346.591234,1
2,"SELECT status, COUNT(*) FROM customer_transact...",67,0,1,0,0,4,579.446077,1
3,SELECT * FROM customer_transactions ORDER BY t...,79,0,0,1,0,100,299.226999,1
4,SELECT * FROM customer_transactions ORDER BY t...,79,0,0,1,0,100,270.514965,1
